# 50 X/Y Concept Probes — Independent AI-Authored Text Validation

[Open in Colab](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/experiments/2026-11/1124_probe_50pairs/qwen_probe_50pairs_validation.ipynb)

**Goal:** Screen the fixed 50 candidate X/Y pairs for decodability in Qwen2.5-0.5B-Instruct before collecting Avoid X / Neutral X effects.

- Training and held-out evaluation are **separate text sets**, generated by OpenAI API with different task instructions, and then checked for lexical overlap and semantic similarity.
- Fixed Qwen checkpoint, assistant-start token, layers **22/23/24**; `StandardScaler` + `LogisticRegression(C=0.1)`.
- Measures: held-out ROC-AUC, leave-one-training-family-out sensitivity, **balanced training-label permutations**, Benjamini–Hochberg FDR.
- Output includes **all 50 pairs** even when unconfirmed. It does not cherry-pick X/Y based on Avoid/Neutral outcomes.
- **Not a human-independent scientific validation:** independent means held-out from training. OpenAI-generated descriptions can share stereotyped factual content, and may produce an artificially easy classification problem. Review examples before drawing conclusions.
- Does **not** run the main suppression experiment; no Avoid X or Neutral X prompts are used here.

**Run instructions:** Google Colab → GPU (T4 or better), enable Secrets `OPENAI_API_KEY` and `GITHUB_RESULTS_TOKEN` (fine-grained Contents read/write on this repository). Run the notebook top to bottom. Results are saved under a unique run ID. Re-running with the same run ID resumes immutable source/checkpoints and verifies configuration; change the run ID if prompts or settings change.

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" "openai>=1.70,<3" "statsmodels>=0.14,<1" pandas numpy requests tqdm matplotlib

In [ ]:
import os, re, io, json, math, time, base64, hashlib, random, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
import numpy as np, pandas as pd, requests, torch
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from google.colab import userdata

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
REPO = "ryouy/GraduationThesis"
SOURCE_CSV = "docs/concept_pair_candidates_50_v0_1.csv"
ROOT = "experiments/2026-11/1124_probe_50pairs"
RUN_ID = "20261124_probe50_independent_v1"
DEST = f"{ROOT}/results/{RUN_ID}"
WORK = Path("/content/qwen_probe_50pairs") / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
API_MODEL = "gpt-4o-mini"   # Change only with a NEW RUN_ID.
LAYERS = [22, 23, 24]
PER_CLASS = 8
SHUFFLES = 499
BATCH_SIZE = 24
PROBE_C = 0.1
AUC_MIN = 0.80
FDR_MAX = 0.05

def secret(name):
    try: value = userdata.get(name)
    except Exception: value = None
    if not value: raise RuntimeError(f"Set Colab Secret {name} and grant notebook access")
    return value

OPENAI = OpenAI(api_key=secret("OPENAI_API_KEY"))
GH_TOKEN = secret("GITHUB_RESULTS_TOKEN")
gh = requests.Session()
gh.headers.update({"Authorization": f"Bearer {GH_TOKEN}",
                   "Accept": "application/vnd.github+json",
                   "X-GitHub-Api-Version": "2022-11-28"})
API_ROOT = f"https://api.github.com/repos/{REPO}/contents/"

def git_meta(p):
    r = gh.get(API_ROOT+p, timeout=90)
    if r.status_code == 404: return None
    r.raise_for_status()
    return r.json()

def git_get(p):
    info = git_meta(p)
    if info is None: return None
    r = gh.get(info["download_url"], timeout=180); r.raise_for_status()
    return r.content

def blob_sha(content):
    return hashlib.sha1(b"blob "+str(len(content)).encode()+b"\\0"+content).hexdigest()

def git_put_immutable(p, content):
    old = git_meta(p)
    if old:
        if old["sha"] != blob_sha(content):
            raise RuntimeError("Immutable result differs: "+p+" — choose a NEW RUN_ID")
        return
    data = {"message": "Freeze Probe50 artifact: "+p.rsplit("/",1)[-1],
            "branch": "main", "content": base64.b64encode(content).decode()}
    r = gh.put(API_ROOT+p,json=data,timeout=180);r.raise_for_status()
    print("Saved",p)

raw = git_get(SOURCE_CSV)
assert raw is not None, "Candidate CSV missing"
pairs = pd.read_csv(io.BytesIO(raw), keep_default_na=False)
assert len(pairs)==50 and pairs.candidate_id.nunique()==50
assert (pairs.selection_status=="CANDIDATE_NOT_FROZEN").all()
names = sorted(set(pairs.target_x)|set(pairs.comparator_y))
print("Pairs:",len(pairs),"unique concept texts:",len(names))
PARAMS = {"run_id":RUN_ID,"source_csv_sha256":hashlib.sha256(raw).hexdigest(),
          "model":MODEL_ID,"revision":REVISION,"api_model":API_MODEL,
          "layers":LAYERS,"per_class":PER_CLASS,"shuffle_count":SHUFFLES,
          "probe_C":PROBE_C,"seed":SEED,"batch_size":BATCH_SIZE,
          "auc_min":AUC_MIN,"fdr_max":FDR_MAX,
          "text_generation_version":"v1_train_eval_distinct_prompts"}
config_raw = json.dumps(PARAMS,sort_keys=True,indent=2).encode()
git_put_immutable(f"{DEST}/config.json",config_raw)
git_put_immutable(f"{DEST}/candidate_pairs.csv",raw)


## 1. Generate description sentences with OpenAI API

Each unique concept receives **8 training** and **8 held-out** sentences. Requests differ by intent, facts, and prompt wording; exact/similar overlap checks follow. These are **AI-authored candidate explanations**, not ground-truth human annotations.

Each concept's responses are saved to GitHub before GPU extraction. The notebook is restartable without re-purchasing the same generations. If generation fails lexical/quality checks, this run stops and prints audit details; it never quietly relabels or discards a failed class.

In [ ]:
def normalized(t):
    return " ".join(re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKC",str(t)).lower()))

def has_label(text, label):
    label = label.strip().lower()
    return bool(re.search(r"(?<![a-z])"+re.escape(label)+r"(?![a-z])", text.lower()))

def generate_set(name, split):
    if split=="train":
        task = ("Produce eight distinct, factual, self-contained English sentences describing"
                " concrete physical properties, construction, parts, or mechanisms of the concept.")
    else:
        task = ("Produce eight distinct, factual, self-contained English sentences about"
                " behaviours, use scenarios, practical situations, environments, or observable consequences."
                " Avoid merely paraphrasing physical descriptions.")
    system = ("You are generating candidate research stimuli for concept probing, not labels."
              " Return JSON ONLY with a key 'sentences' containing exactly eight sentences."
              " Each sentence must be 9-25 words, independent, natural, and factual."
              " Never include the concept name, obvious aliases, or the name of another item being contrasted."
              " Do not use pronouns requiring another sentence. Do not number or label the sentences."
              " Avoid metaphorical or subjective claims. Vary facts, grammar and word choice.")
    user = f"Concept: {name}\\n{task}"
    completion = OPENAI.chat.completions.create(
        model=API_MODEL, temperature=0.7,
        response_format={"type":"json_object"},
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    data=json.loads(completion.choices[0].message.content)
    sentences=data["sentences"]
    if not isinstance(sentences,list) or len(sentences)!=PER_CLASS:
        raise ValueError(f"{name}/{split}: API returned wrong count")
    if any(not isinstance(s,str) or len(s.split())<7 for s in sentences):
        raise ValueError(f"{name}/{split}: malformed sentences")
    if any(has_label(s,name) for s in sentences):
        bad=[s for s in sentences if has_label(s,name)]
        raise ValueError(f"{name}/{split}: label leakage: {bad}")
    if len(set(map(normalized,sentences)))!=len(sentences):
        raise ValueError(f"{name}/{split}: duplicate sentences")
    return sentences

all_texts={}
for name in tqdm(names,desc="Prepare unique concept descriptions"):
    key=hashlib.sha256(name.encode()).hexdigest()[:12]
    remote=f"{DEST}/texts/concept_{key}.json"
    prior=git_get(remote)
    if prior is not None:
        datum=json.loads(prior)
        assert datum["concept"]==name and datum["params"]==PARAMS
    else:
        datum={"concept":name,"params":PARAMS,"created_at":datetime.now(timezone.utc).isoformat(),
               "train":generate_set(name,"train"),"eval":generate_set(name,"eval")}
        git_put_immutable(remote,json.dumps(datum,ensure_ascii=False,sort_keys=True,indent=2).encode())
    all_texts[name]=datum
print("Generated/recovered:",len(all_texts),"concepts")


## 2. Verify lexical separation and inspect possible factual overlap

- Neither target name nor its comparator name may appear in **either** X or Y class text.
- Exact train/eval duplicates and close `SequenceMatcher` text matches are blocked.
- We **flag** similarities that can still be semantically redundant; human spot review is still recommended.
- We do not use the Avoid/Neutral experimental conditions during this verification.

In [ ]:
audit=[];stim=[]
for name in names:
    datum=all_texts[name]
    for split in ("train","eval"):
        for i,text in enumerate(datum[split]):
            stim.append({"concept":name,"split":split,"family":i,"user_prompt":text})
texts=pd.DataFrame(stim)
assert len(texts)==len(names)*2*PER_CLASS
for row in pairs.itertuples():
    x,y=row.target_x,row.comparator_y
    relevant=texts[texts.concept.isin([x,y])]
    hits=[(rec.concept,rec.split,rec.family,word) for rec in relevant.itertuples()
          for word in (x,y) if has_label(rec.user_prompt,word)]
    audit.append({"candidate_id":row.candidate_id,"target_x":x,"comparator_y":y,
                  "n_lexical_label_hits":len(hits),"label_hits":json.dumps(hits),
                  "lexically_valid":len(hits)==0})
audit=pd.DataFrame(audit)
overlap=[]
for name in names:
    a=all_texts[name]["train"];b=all_texts[name]["eval"]
    for i,t in enumerate(a):
        for j,u in enumerate(b):
            similarity=SequenceMatcher(None,normalized(t),normalized(u)).ratio()
            if similarity>=0.70:
                overlap.append({"concept":name,"train_family":i,"eval_family":j,
                                "similarity":similarity,"train":t,"eval":u})
lex_bad=audit[~audit.lexically_valid]
print("Name-leak pairs:",len(lex_bad),"surface-similarity flags:",len(overlap))
display(lex_bad.head(20))
if overlap: display(pd.DataFrame(overlap).head(20))
# Record all text even if it fails; never silently exclude a pair.
texts.to_csv(WORK/"all_descriptions.csv",index=False)
audit.to_csv(WORK/"pair_lexical_audit.csv",index=False)
pd.DataFrame(overlap,columns=["concept","train_family","eval_family","similarity","train","eval"]).to_csv(WORK/"train_eval_overlap_flags.csv",index=False)
git_put_immutable(f"{DEST}/all_descriptions.csv",(WORK/"all_descriptions.csv").read_bytes())
git_put_immutable(f"{DEST}/pair_lexical_audit.csv",(WORK/"pair_lexical_audit.csv").read_bytes())
git_put_immutable(f"{DEST}/train_eval_overlap_flags.csv",(WORK/"train_eval_overlap_flags.csv").read_bytes())
assert len(lex_bad)==0, "Lexical leakage detected. Review flagged rows; use NEW RUN_ID for regenerated stimuli."
assert len(overlap)==0, "Close train/eval texts detected. Review them; use NEW RUN_ID for revision."
print("Surface audits passed. Semantic independence is NOT guaranteed.")


## 3. Extract Qwen hidden states (assistant prefix, L22/23/24)

This reuses the location and model revision of the prior study. Checkpoint chunks are immutable in GitHub. Hidden states are calculated once for each sentence (not once per directed candidate pair).

In [ ]:
assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,revision=REVISION,torch_dtype=torch.float32,
    attn_implementation="eager").to("cuda").eval()
assert model.config.num_hidden_layers==24
@torch.inference_mode()
def extract(text):
    ids=tokenizer.apply_chat_template(
        [{"role":"user","content":text}],tokenize=True,
        add_generation_prompt=True,return_tensors="pt").to("cuda")
    hs=model(input_ids=ids,output_hidden_states=True,use_cache=False).hidden_states
    return np.stack([hs[l][0,-1].float().cpu().numpy() for l in LAYERS])

text_raw=(WORK/"all_descriptions.csv").read_bytes()
input_hash=hashlib.sha256(text_raw).hexdigest()
n=len(texts);n_batches=math.ceil(n/BATCH_SIZE)
features=[]
for b in tqdm(range(n_batches),desc="Qwen hidden states"):
    start,end=b*BATCH_SIZE,min(n,(b+1)*BATCH_SIZE)
    remote=f"{DEST}/features/chunk_{b:03d}.npz"
    saved=git_get(remote)
    if saved is None:
        chunk=np.stack([extract(t) for t in texts.user_prompt.iloc[start:end]])
        buf=io.BytesIO()
        np.savez_compressed(buf,features=chunk,begin=np.int64(start),end=np.int64(end),
                            input_hash=np.array(input_hash),layers=np.array(LAYERS))
        saved=buf.getvalue()
        git_put_immutable(remote,saved)
    with np.load(io.BytesIO(saved),allow_pickle=False) as f:
        assert int(f["begin"])==start and int(f["end"])==end
        assert str(f["input_hash"])==input_hash
        assert f["layers"].tolist()==LAYERS
        x=f["features"].copy()
        assert x.shape==(end-start,3,model.config.hidden_size) and np.isfinite(x).all()
        features.append(x)
vectors=np.concatenate(features)
assert len(vectors)==len(texts)
indices={(r.concept,r.split,r.family):i for i,r in enumerate(texts.itertuples())}
def values(name,split,layer_idx):
    return np.stack([vectors[indices[name,split,i],layer_idx] for i in range(PER_CLASS)])
print("Verified tensor",vectors.shape)


## 4. Frozen X/Y Probe fitting and independent evaluation

Each pair is trained only with its own X and Y train examples. The evaluator uses their held-out examples. No hyperparameter tuning uses held-out AUC; the adoption threshold is a **candidate screening criterion** and is not proof of causal suppression.

Permutation null refits the classifier on **class-balanced shuffled training labels**. Its p-values are diagnostic given small, AI-generated text sets. FDR is calculated across **all 50 directed candidate pairs**.

In [ ]:
def fit_probe(a,b,labels=None):
    x=np.concatenate([a,b])
    labels=np.r_[np.ones(len(a),dtype=int),np.zeros(len(b),dtype=int)] if labels is None else labels
    scaler=StandardScaler().fit(x)
    clf=LogisticRegression(C=PROBE_C,solver="liblinear",max_iter=1000,random_state=SEED).fit(scaler.transform(x),labels)
    return scaler,clf
def auc(scaler,clf,a,b):
    score=clf.decision_function(scaler.transform(np.concatenate([a,b])))
    labels=np.r_[np.ones(len(a),dtype=int),np.zeros(len(b),dtype=int)]
    return float(roc_auc_score(labels,score))
def stable_seed(s):
    return int(hashlib.sha256((s+str(SEED)).encode()).hexdigest()[:8],16)

results=[]
for p in tqdm(pairs.itertuples(),total=len(pairs),desc="50 independent Probe screens"):
    row={"candidate_id":p.candidate_id,"category":p.category,"target_x":p.target_x,
         "comparator_y":p.comparator_y,"prior_evidence":p.prior_evidence}
    for li,l in enumerate(LAYERS):
        a,b=values(p.target_x,"train",li),values(p.comparator_y,"train",li)
        ea,eb=values(p.target_x,"eval",li),values(p.comparator_y,"eval",li)
        scaler,clf=fit_probe(a,b)
        row[f"L{l}_auc"]=auc(scaler,clf,ea,eb)
        if l==24:
            loo=[]
            for omit in range(PER_CLASS):
                keep=[j for j in range(PER_CLASS) if j!=omit]
                ss,cc=fit_probe(a[keep],b[keep])
                loo.append(auc(ss,cc,ea,eb))
            row["L24_loo_min"]=min(loo);row["L24_loo_mean"]=float(np.mean(loo))
            rng=np.random.default_rng(stable_seed(p.candidate_id))
            x=np.concatenate([a,b])
            y=np.r_[np.ones(PER_CLASS,dtype=int),np.zeros(PER_CLASS,dtype=int)]
            zx=scaler.transform(x)
            ze=scaler.transform(np.concatenate([ea,eb]))
            null=[]
            for _ in range(SHUFFLES):
                fake=rng.permutation(y)
                cc=LogisticRegression(C=PROBE_C,solver="liblinear",max_iter=1000,random_state=SEED).fit(zx,fake)
                null.append(roc_auc_score(y,cc.decision_function(ze)))
            row["p_shuffle_plus_one"]=(1+sum(v>=row["L24_auc"]-1e-12 for v in null))/(SHUFFLES+1)
            row["shuffle_mean_auc"]=float(np.mean(null))
    results.append(row)
scores=pd.DataFrame(results)
scores["fdr_q"]=multipletests(scores.p_shuffle_plus_one.to_numpy(),method="fdr_bh")[1]
scores["auc_gate"]=scores.L24_auc>=AUC_MIN
scores["fdr_gate"]=scores.fdr_q<=FDR_MAX
scores["screen_pass"]=scores.auc_gate & scores.fdr_gate
scores["selection_status"]=np.where(scores.screen_pass,"SCREEN_PASS_REVIEW_REQUIRED","SCREEN_FAIL")
scores["new_validation_completed"]=True
scores.to_csv(WORK/"pair_probe_scores.csv",index=False)
display(scores[["candidate_id","target_x","comparator_y","L24_auc","L24_loo_min","fdr_q","screen_pass"]])
print("Screen pass:",int(scores.screen_pass.sum()),"/",len(scores))


## 5. Summary, complete artifacts and next decision

This is a **screening experiment**, not adoption to the Avoid/Neutral main experiment. Inspect factual claims, name leakage and label overlap, especially for high-AUC pairs, before freezing weights and target inventory.

Every pair's results are published, including failed pairs. Source descriptions and hidden-state checkpoints are retained to reproduce the fit. Do not modify or re-run the same run ID with changed prompts or configuration.

In [ ]:
summary=[
"# Probe50 independent-text screen",
"",
f"- Run ID: `{RUN_ID}`",
f"- Model: `{MODEL_ID}` @ `{REVISION}`",
f"- Candidate pairs: **{len(scores)}** (none selected using suppression outputs)",
f"- Unique concept text banks: **{len(names)}**",
f"- Description rows: **{len(texts)}** (train and held-out separately)",
f"- AUC≥{AUC_MIN} and BH-q≤{FDR_MAX}: **{int(scores.screen_pass.sum())} / {len(scores)}**",
f"- Median L24 held-out AUC: **{scores.L24_auc.median():.3f}**",
"",
"**These screens are preliminary**: the data were AI-authored, not independently verified by humans.",
"High AUC means X/Y discriminability of held-out descriptions, not the strength of thinking about X.",
"Failure cases are retained; no threshold or candidate pair should be changed after viewing Avoid/Neutral responses.",
"",
"## Per-category",
"",
"| Category | Pairs | Screen pass |",
"|---|---:|---:|",
]
for cat,g in scores.groupby("category",sort=True):
    summary.append(f"| {cat} | {len(g)} | {int(g.screen_pass.sum())} |")
(WORK/"analysis_summary.md").write_text("\\n".join(summary)+"\\n",encoding="utf-8")
manifest={"status":"SCREEN_COMPLETED_HUMAN_REVIEW_PENDING","timestamp":datetime.now(timezone.utc).isoformat(),
          "parameters":PARAMS,"input_sha256":input_hash,
          "n_pairs":len(scores),"n_names":len(names),"n_prompts":len(texts),
          "n_screen_pass":int(scores.screen_pass.sum()),
          "source":SOURCE_CSV,
          "limitations":["AI-authored labels; no independent human fact audit",
                         "AUC does not measure mental activation",
                         "Pair preselection and 50 comparisons; exploratory screen"]}
(WORK/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+"\\n")
for f in ("pair_probe_scores.csv","analysis_summary.md","manifest.json"):
    git_put_immutable(f"{DEST}/{f}",(WORK/f).read_bytes())
print((WORK/"analysis_summary.md").read_text())
print("All outputs and checkpoints saved to:",DEST)
